# Kairos Inference Benchmark Suite

Evaluates pre-trained Kairos on **6 standard test benchmarks** (no training):
1. GSM8K (test) — Math reasoning
2. ARC-Challenge (test) — Logic
3. BBH — BIG-Bench Hard reasoning
4. MATH-500 — Competition math
5. HumanEval — Code generation
6. MBPP (test) — Code generation

In [ ]:
!pip install -q torch transformers datasets accelerate torchdiffeq bitsandbytes
import subprocess, sys, os
repo = 'CIDA-plugin'
if not os.path.exists(repo):
    subprocess.run(['git','clone','--depth','1',
                   'https://github.com/Kairatzh/CIDA-plugin.git'], check=True)
sys.path.insert(0, os.path.abspath(repo))
from google.colab import drive
drive.mount('/content/drive')


## 0. Load Frozen LLM + Kairos Weights

In [ ]:
import gc, re, json, time, random, torch, torch.nn as nn
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm.auto import tqdm
from cida_plugin import CIDAPlugin, CIDAPluginConfig

DEVICE = "cuda"
# To test the new continual fine-tuned weights, use 'continual_best/best'
WEIGHTS_PATH = '/content/drive/MyDrive/Kairos_CIDA_7B/continual_best/best'
# To test the original math-only weights, use:
# WEIGHTS_PATH = '/content/drive/MyDrive/Kairos_CIDA_7B/best'

def flush():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

info_path = Path(WEIGHTS_PATH) / "train_info.json"
kcfg_path = Path(WEIGHTS_PATH) / "kairos_config.json"
if info_path.exists():
    with open(info_path) as f: info = json.load(f)
elif kcfg_path.exists():
    with open(kcfg_path) as f: info = json.load(f)
else:
    info = {}

MODEL_NAME = info.get("llm_model_name", "Qwen/Qwen1.5-7B-Chat")
NUM_PLAN_TOKENS = info.get("num_plan_tokens", 8)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model in 8-bit to avoid CUDA OOM on T4 GPU (takes ~7.5GB VRAM instead of 14GB)
print(f"Loading {MODEL_NAME} in 8-bit...")
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True, torch_dtype=torch.float16,
    device_map="auto", load_in_8bit=True, output_hidden_states=True)
for p in llm.parameters(): p.requires_grad = False
llm.eval()

LLM_HIDDEN = llm.config.hidden_size
embed_tokens = llm.get_input_embeddings()
for p in embed_tokens.parameters(): p.requires_grad = False

# CIDA
cida_cfg = CIDAPluginConfig.from_pretrained(WEIGHTS_PATH)
cida = CIDAPlugin(cida_cfg)
sd = torch.load(str(Path(WEIGHTS_PATH)/"pytorch_model.bin"), map_location="cpu")
cida.load_state_dict(sd); del sd; flush()
cida = cida.to(DEVICE).float().eval()

# PlanProjector
class CIDAPlanProjector(nn.Module):
    def __init__(self, d_cida, d_llm, num_agents, num_plan_tokens=8):
        super().__init__()
        self.num_plan_tokens = num_plan_tokens
        self.d_llm = d_llm
        d_flat = num_agents * d_cida
        self.projector = nn.Sequential(
            nn.Linear(d_flat, d_flat*2), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(d_flat*2, num_plan_tokens*d_llm),
            nn.LayerNorm(num_plan_tokens*d_llm))
    def forward(self, state_s):
        B = state_s.size(0)
        return self.projector(state_s.reshape(B,-1).float()).view(B, self.num_plan_tokens, self.d_llm)

proj = CIDAPlanProjector(d_cida=cida_cfg.d_hidden, d_llm=LLM_HIDDEN,
    num_agents=cida_cfg.num_agents, num_plan_tokens=NUM_PLAN_TOKENS).to(DEVICE)
proj.load_state_dict(torch.load(str(Path(WEIGHTS_PATH)/"plan_projector.pt"), map_location=DEVICE))
proj.eval(); flush()
print(f"Loaded: {MODEL_NAME} (8-bit) + Kairos ({NUM_PLAN_TOKENS} plan tokens)")


## Generation Helpers

In [ ]:
@torch.no_grad()
def gen_base(prompt, max_new=128):
    flush()
    tokenizer.padding_side = "left"
    enc = tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True).to(DEVICE)
    ids = llm.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
        max_new_tokens=max_new, do_sample=False,
        pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    plen = enc["input_ids"].size(1)
    text = tokenizer.decode(ids[0, plen:], skip_special_tokens=True)
    del enc, ids; flush()
    return text

@torch.no_grad()
def gen_kairos(prompt, max_new=128):
    flush()
    tokenizer.padding_side = "left"
    enc = tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True).to(DEVICE)
    out = llm(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
              output_hidden_states=True, use_cache=False)
    last_h = out.hidden_states[-1]
    sl = enc["attention_mask"].sum(dim=1) - 1
    pooled = last_h[0, sl[0]].float().unsqueeze(0)
    p_emb = embed_tokens(enc["input_ids"]).detach().float()
    del out, last_h; flush()

    co = cida(pooled=pooled)
    plan = proj(co["state_final"].s)
    cat = torch.cat([p_emb, plan], dim=1).to(next(llm.parameters()).dtype)
    del p_emb, plan, pooled; flush()
    pm = torch.ones(1, NUM_PLAN_TOKENS, dtype=enc["attention_mask"].dtype, device=DEVICE)
    gm = torch.cat([enc["attention_mask"], pm], dim=1)
    ids = llm.generate(inputs_embeds=cat, attention_mask=gm,
        max_new_tokens=max_new, do_sample=False,
        pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(ids[0], skip_special_tokens=True)
    del enc, ids, cat, gm, co; flush()
    return text

def normalise(t):
    t = t.strip().replace(",","")
    try: return str(int(float(t)))
    except: return t

def parse_num(text):
    m = re.search(r"####\s*([\-0-9,\.]+)", text)
    if m: return normalise(m.group(1))
    m = re.search(r"\\boxed\{([^}]+)\}", text)
    if m: return m.group(1).strip()
    nums = re.findall(r"\b-?\d+(?:\.\d+)?\b", text.replace(",",""))
    return normalise(nums[-1]) if nums else ""

def chat(q, sys_msg="You are a mathematical reasoner. Solve step by step, then give the final answer after '####'."):
    return f"<|im_start|>system\n{sys_msg}\n<|im_end|>\n<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"

print("Helpers ready.")


## 1. GSM8K (test) — 1319 examples\n\nThe gold-standard math reasoning benchmark.

In [ ]:
print("Loading GSM8K test...")
gsm = list(load_dataset("openai/gsm8k", "main", split="test"))
print(f"  {len(gsm)} examples")

N = 50  # Set to 50 for a fast run, or len(gsm) for full evaluation
random.seed(42)
samples = random.sample(gsm, min(N, len(gsm)))
print(f"Evaluating {len(samples)} samples...")

base_ok, kairos_ok = 0, 0
for i, ex in enumerate(tqdm(samples, desc="GSM8K")):
    gold = re.search(r"####\s*([\-0-9,\.]+)", ex["answer"])
    gold_num = normalise(gold.group(1)) if gold else ""
    prompt = chat(ex["question"])

    # Base
    bt = gen_base(prompt, max_new=128)
    bp = parse_num(bt)
    if bp == gold_num: base_ok += 1

    # Kairos
    kt = gen_kairos(prompt, max_new=128)
    kp = parse_num(kt)
    if kp == gold_num: kairos_ok += 1

    if (i+1) % 10 == 0:
        print(f"  [{i+1}/{len(samples)}]  Base: {base_ok/(i+1)*100:.1f}%  Kairos: {kairos_ok/(i+1)*100:.1f}%")

gsm_base = base_ok/len(samples)*100
gsm_kairos = kairos_ok/len(samples)*100
print(f"\nGSM8K FINAL:  Base={gsm_base:.2f}%  Kairos={gsm_kairos:.2f}%  Delta={gsm_kairos-gsm_base:+.2f} pp")


## 2. ARC-Challenge (test) — 1172 examples

In [ ]:
print("Loading ARC-Challenge test...")
arc = list(load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test"))
print(f"  {len(arc)} examples")

N_ARC = 50  # Set to 50 for a fast run, or len(arc) for full evaluation
random.seed(42)
arc_samples = random.sample(arc, min(N_ARC, len(arc)))
print(f"Evaluating {len(arc_samples)} samples...")

arc_sys = "You are a logical reasoner. Pick the correct answer (A, B, C, D, or E). State ONLY the letter."

base_ok, kairos_ok = 0, 0
for i, ex in enumerate(tqdm(arc_samples, desc="ARC")):
    choices = ex["choices"]
    opts = "\n".join([f"{l}) {t}" for l, t in zip(choices["label"], choices["text"])])
    q = f"{ex['question']}\n{opts}"
    gold = ex["answerKey"].strip().upper()
    prompt = chat(q, arc_sys)

    bt = gen_base(prompt, max_new=32)
    bp = re.search(r"([A-E])", bt.strip()[:10])
    bp = bp.group(1) if bp else ""
    if bp == gold: base_ok += 1

    kt = gen_kairos(prompt, max_new=32)
    kp = re.search(r"([A-E])", kt.strip()[:10])
    kp = kp.group(1) if kp else ""
    if kp == gold: kairos_ok += 1

    if (i+1) % 10 == 0:
        print(f"  [{i+1}/{len(arc_samples)}]  Base: {base_ok/(i+1)*100:.1f}%  Kairos: {kairos_ok/(i+1)*100:.1f}%")

arc_base = base_ok/len(arc_samples)*100
arc_kairos = kairos_ok/len(arc_samples)*100
print(f"\nARC FINAL:  Base={arc_base:.2f}%  Kairos={arc_kairos:.2f}%  Delta={arc_kairos-arc_base:+.2f} pp")


## 3. BIG-Bench Hard (BBH) — Selected Subtasks

In [ ]:
print("Loading BBH subtasks...")
bbh_tasks = ["logical_deduction_three_objects", "navigate", "date_understanding",
             "tracking_shuffled_objects_three_objects", "web_of_lies"]
bbh_sys = "You are a logical reasoner. Think step by step and give the final answer."

N_BBH = 10  # 10 samples per subtask (50 total) for a fast run, or 50/100 for full evaluation

all_base, all_kairos, all_n = 0, 0, 0
for task_name in bbh_tasks:
    try:
        ds = list(load_dataset("lukaemon/bbh", task_name, split="test"))
    except:
        print(f"  Skipping {task_name} (not found)")
        continue
    n = min(N_BBH, len(ds))
    random.seed(42)
    samp = random.sample(ds, n)
    b_ok, k_ok = 0, 0
    for ex in tqdm(samp, desc=task_name[:20]):
        q = ex["input"]
        gold = ex["target"].strip().lower()
        prompt = chat(q, bbh_sys)

        bt = gen_base(prompt, max_new=64)
        if gold in bt.strip().lower()[-60:]: b_ok += 1

        kt = gen_kairos(prompt, max_new=64)
        if gold in kt.strip().lower()[-60:]: k_ok += 1

    print(f"  {task_name}: Base={b_ok}/{n} ({b_ok/n*100:.0f}%)  Kairos={k_ok}/{n} ({k_ok/n*100:.0f}%)")
    all_base += b_ok; all_kairos += k_ok; all_n += n

bbh_base = all_base/all_n*100 if all_n else 0
bbh_kairos = all_kairos/all_n*100 if all_n else 0
print(f"\nBBH FINAL:  Base={bbh_base:.2f}%  Kairos={bbh_kairos:.2f}%  Delta={bbh_kairos-bbh_base:+.2f} pp")


## 4. MATH-500 — Competition Math

In [ ]:
print("Loading MATH-500...")
try:
    math_ds = list(load_dataset("HuggingFaceH4/MATH-500", split="test"))
except Exception as e:
    print(f"Failed to load HuggingFaceH4/MATH-500: {e}")
    print("Falling back to local dummy dataset to prevent crashing...")
    math_ds = [{"problem": "Solve for x: x + 2 = 5", "solution": "x = 3", "answer": "3"}]

N_MATH = 50  # Take 50 random samples for a fast run, or len(math_ds) for full evaluation
random.seed(42)
math_samples = random.sample(math_ds, min(N_MATH, len(math_ds)))
print(f"Evaluating {len(math_samples)} samples...")

math_sys = "You are a competition math solver. Show your work step by step. Put the final answer inside \\boxed{}."

base_ok, kairos_ok = 0, 0
for i, ex in enumerate(tqdm(math_samples, desc="MATH-500")):
    q = ex["problem"]
    gold = ex["answer"].strip()
    prompt = chat(q, math_sys)

    bt = gen_base(prompt, max_new=128)
    bm = re.search(r"\\boxed\{([^}]+)\}", bt)
    bp = bm.group(1).strip() if bm else parse_num(bt)
    if bp == gold or normalise(bp) == normalise(gold): base_ok += 1

    kt = gen_kairos(prompt, max_new=128)
    km = re.search(r"\\boxed\{([^}]+)\}", kt)
    kp = km.group(1).strip() if km else parse_num(kt)
    if kp == gold or normalise(kp) == normalise(gold): kairos_ok += 1

    if (i+1) % 10 == 0:
        print(f"  [{i+1}/{len(math_samples)}]  Base: {base_ok/(i+1)*100:.1f}%  Kairos: {kairos_ok/(i+1)*100:.1f}%")

math_base = base_ok/len(math_samples)*100
math_kairos = kairos_ok/len(math_samples)*100
print(f"\nMATH-500 FINAL:  Base={math_base:.2f}%  Kairos={math_kairos:.2f}%  Delta={math_kairos-math_base:+.2f} pp")


## 5. HumanEval — Code Generation (pass@1)

In [ ]:
print("Loading HumanEval...")
he = list(load_dataset("openai/openai_humaneval", split="test"))
print(f"  {len(he)} problems")

N_HE = 50  # Take 50 random samples for a fast run, or len(he) for full evaluation
random.seed(42)
he_samples = random.sample(he, min(N_HE, len(he)))
print(f"Evaluating {len(he_samples)} samples...")

code_sys = "You are a Python expert. Complete the function. Output ONLY the Python code, no explanations."

base_ok, kairos_ok = 0, 0
for i, ex in enumerate(tqdm(he_samples, desc="HumanEval")):
    prompt_text = ex["prompt"]
    test_code = ex["test"]
    entry = ex["entry_point"]
    p = chat(f"Complete this Python function:\n```python\n{prompt_text}\n```", code_sys)

    for label, gen_fn in [("base", gen_base), ("kairos", gen_kairos)]:
        raw = gen_fn(p, max_new=256)
        # Extract code block if present
        cm = re.search(r"```python\n(.*?)```", raw, re.DOTALL)
        code_str = cm.group(1) if cm else raw
        # Prepend prompt if function def not present
        if f"def {entry}" not in code_str:
            code_str = prompt_text + "\n" + code_str
        # Try executing with test cases
        try:
            exec_globals = {}
            exec(code_str + "\n" + test_code, exec_globals)
            # If no exception, test passed
            if label == "base": base_ok += 1
            else: kairos_ok += 1
        except:
            pass

    if (i+1) % 10 == 0:
        print(f"  [{i+1}/{len(he_samples)}]  Base: {base_ok/(i+1)*100:.1f}%  Kairos: {kairos_ok/(i+1)*100:.1f}%")

he_base = base_ok/len(he_samples)*100
he_kairos = kairos_ok/len(he_samples)*100
print(f"\nHumanEval FINAL:  Base={he_base:.2f}%  Kairos={he_kairos:.2f}%  Delta={he_kairos-he_base:+.2f} pp")


## 6. MBPP (test) — Code Generation

In [ ]:
print("Loading MBPP test...")
mbpp = list(load_dataset("google-research-datasets/mbpp", split="test"))
print(f"  {len(mbpp)} problems")

N_MBPP = 50  # Take 50 random samples for a fast run, or len(mbpp) for full evaluation
random.seed(42)
mbpp_samples = random.sample(mbpp, min(N_MBPP, len(mbpp)))
print(f"Evaluating {len(mbpp_samples)} samples...")

base_ok, kairos_ok = 0, 0
for i, ex in enumerate(tqdm(mbpp_samples, desc="MBPP")):
    q = ex["text"]
    tests = ex["test_list"]
    p = chat(f"Write a Python function for: {q}", code_sys)

    for label, gen_fn in [("base", gen_base), ("kairos", gen_kairos)]:
        raw = gen_fn(p, max_new=256)
        cm = re.search(r"```python\n(.*?)```", raw, re.DOTALL)
        code_str = cm.group(1) if cm else raw
        try:
            exec_globals = {}
            exec(code_str, exec_globals)
            for tc in tests:
                exec(tc, exec_globals)
            if label == "base": base_ok += 1
            else: kairos_ok += 1
        except:
            pass

    if (i+1) % 10 == 0:
        print(f"  [{i+1}/{len(mbpp_samples)}]  Base: {base_ok/(i+1)*100:.1f}%  Kairos: {kairos_ok/(i+1)*100:.1f}%")

mbpp_base = base_ok/len(mbpp_samples)*100
mbpp_kairos = kairos_ok/len(mbpp_samples)*100
print(f"\nMBPP FINAL:  Base={mbpp_base:.2f}%  Kairos={mbpp_kairos:.2f}%  Delta={mbpp_kairos-mbpp_base:+.2f} pp")


## Final Results Summary

In [ ]:
print("=" * 75)
print(f"  {'Benchmark':<20} {'N':>5}  {'Base LLM':>10}  {'Kairos':>10}  {'Delta':>10}")
print("=" * 75)
benchmarks = [
    ("GSM8K (Math)", len(samples), gsm_base, gsm_kairos),
    ("ARC-Challenge", len(arc_samples), arc_base, arc_kairos),
    ("BBH (Reasoning)", all_n, bbh_base, bbh_kairos),
    ("MATH-500", len(math_samples), math_base, math_kairos),
    ("HumanEval (Code)", len(he_samples), he_base, he_kairos),
    ("MBPP (Code)", len(mbpp_samples), mbpp_base, mbpp_kairos),
]
for name, n, b, k in benchmarks:
    d = k - b
    print(f"  {name:<20} {n:>5}  {b:>9.2f}%  {k:>9.2f}%  {d:>+9.2f} pp")
print("=" * 75)

avg_base = sum(b for _,_,b,_ in benchmarks) / len(benchmarks)
avg_kairos = sum(k for _,_,_,k in benchmarks) / len(benchmarks)
print(f"  {'AVERAGE':<20} {'':>5}  {avg_base:>9.2f}%  {avg_kairos:>9.2f}%  {avg_kairos-avg_base:>+9.2f} pp")
print()
print(f"  Model: {MODEL_NAME} + Kairos CIDA Plugin")
print(f"  Trainable params: CIDA only (LLM frozen)")
